# Solution of a hydrodynamic system using the convolution integral
This notebook demonstrates solving the motion of a hydrodynamic system, calculating power, and evaluating the gradient of the system power with respect to PTO damping.

1. Define or ingest hydrodynamic coefficients
2. Define numerical simulation parameters and wave conditions
3. Define body mass properties, initial conditions, and PTO damping
4. Solve the system and calculate power
5. Gradient of power with respect to PTO damping

In [1]:
# First import all necessary julia packages:
using Unitful
using DimensionfulAngles: radᵃ as rad, θ₀, 𝐀, Dispersion
using LinearAlgebra
using Plots
import Random
# import Statistics as Stats
import ForwardDiff as FD
import WaveSpectra

include("..\\src\\Hydrodynamics.jl")

Main.Hydrodynamics

In [2]:
# Hydrodynamics
hydro = Hydrodynamics.Bemio.read_capytaine("data//rm3.nc")

addedMassUnits = 1.0 .* [repeat([u"kg"], 3, 3) repeat([0u"kg*m"/rad], 3, 3); repeat([0u"kg*m"], 3, 3) repeat([0u"kg*m^2"/rad], 3, 3)]
addedMassCoeff = hydro.am[1:6, 1:6, :] .* addedMassUnits # Dimensions: influenced_dof radiating_dof omega

# Don't define force units with Newtons -- it will end up causing conflicts in the out-of-place ODE solve (OOP is required for Zygote)
radDampingUnits = 1.0 * [repeat([u"kg*m/s^2 / (m/s)"], 3, 3) repeat([u"kg*m/s^2 / (1/s)"/rad], 3, 3); repeat([u"kg*m/s^2 * m / (m/s)"], 3, 3) repeat([u"kg*m/s^2 * m / (1/s)"/rad], 3, 3)]
radDampingCoeff = hydro.rd[1:6, 1:6, :] .* radDampingUnits # Dimensions: influenced_dof radiating_dof omega

khsUnits = 1.0 .* [repeat([u"kg*m/s^2 / m"], 3, 3) repeat([u"kg*m/s^2"/rad], 3, 3); repeat([u"kg*m/s^2 * m / m"], 3, 3) repeat([u"kg*m/s^2 * m"/rad], 3, 3)]
khsCoeff = hydro.khs[1:6, 1:6] .* khsUnits # Dimensions: influenced_dof radiating_dof

exUnits = [repeat([u"kg*m/s^2 / m"], 3, 1); repeat([u"kg*m/s^2 * m / m"], 3, 1)]
exCoeff = hydro.ex[1:6, :, :, :] .* exUnits # Dimensions: influenced_dof wave_dir omega complex

Kᵣ, tᵣ = Hydrodynamics.Bemio.radiation_irf(hydro.rd[1:6, 1:6, :], hydro.w, w_max = 5.0, t_f = 30.0, dt = 0.01)
tᵣ = tᵣ .* u"s"
Kᵣ = Kᵣ .* radDampingUnits .* u"s^-1"
addedMassInf = addedMassCoeff[:,:,end]
# addedMassInf = Hydrodynamics.Bemio.alternate_Ainf(Unitful.ustrip.(Kᵣ), hydro.am, hydro.w, Unitful.ustrip.(tᵣ)) .* addedMassUnits

6×6 Matrix{Quantity{Float64}}:
 62581.9 kg              4.08645e-7 kg   …  -0.0 kg m rad^-1
    -6.27576e-7 kg   61535.7 kg              0.0 kg m rad^-1
     8.77205e-11 kg      1.50686e-11 kg     -0.0 kg m rad^-1
     0.0 kg m           -0.0 kg m           -0.0 kg m^2 rad^-1
     0.0 kg m            0.0 kg m           -0.0 kg m^2 rad^-1
    -0.0 kg m           -0.0 kg m        …   0.0 kg m^2 rad^-1

### Wave conditions and numerical set-up
Define relevant wave using WaveSpectra representations (including significant wave height, energy period, spectra, frequency dimension, etc) and time marching parameters (start, ramp, end times; time step)

In [3]:
# Wave conditions
significant_waveheight = 1.0u"m"
amplitude = significant_waveheight/2
energy_period = 6.0u"s"

omega = hydro.w*rad*u"1/s"
dOmega = omega[2] - omega[1]

frequency = uconvert.(u"s^-1", omega, Dispersion())
dFrequency = uconvert.(u"s^-1", dOmega, Dispersion())

spectrum = WaveSpectra.ParametricSpectra.spectrum_pierson_moskowitz(frequency, significant_waveheight, energy_period)
peak_omega_index = argmax(spectrum)
phase = rand(Random.Xoshiro(0), Float64, size(spectrum)) * 2 * pi * rad

# Time-domain set-up
# dt = 10u"ms" # results in type Rational when combined used in the `collect` call
dt = 1.0e-1u"s"
t0 = 0.0u"s"
tf = energy_period * 10
ts = collect(t0:dt:tf)
nt = length(ts)
ramp_time = 6.0*1u"s"
i_ramp = Int64(ramp_time / dt + 1)

elevation = sum(sqrt.(2*spectrum*dFrequency) .* cos.(omega.*ts' .+ phase), dims=1)

1×601 Matrix{Quantity{Float64, 𝐋, Unitful.FreeUnits{(m,), 𝐋, nothing}}}:
 0.265498 m  0.281374 m  0.2966 m  …  0.141475 m  0.18647 m  0.228464 m

### Define body properties
The required body properties include center of gravity and center of buoyancy, mass, moments and products of inertia, displaced volume, and any initial displacements of the bodies.

PTO damping is lumped with radiation damping when the system of equations is solved, but it is defined as a separate input parameter so that it is a distinct input to power production and can be optimized accordingly.

In [4]:
# Body properties
cg = [0., 0., -3.5]u"m"
m = 725833u"kg"
Ixx = [20907301, 21306090.66, 37085481.11]u"kg*m^2"/rad
Ixy = [1e6, 1e6, 1e5]u"kg*m^2"/rad
I = diagm(Ixx)
I[1, 2:3] = I[2:3, 1] = Ixy[1:2]
I[2, 3] = I[3, 2] = Ixy[3]
body_mass = [diagm(repeat([m], 3)) repeat([0u"kg*m"/rad], 3, 3); repeat([0u"kg*m"], 3, 3) I]

# Coefficients for the form: ẍ + c * ẋ + k * x = 0
dof_names = ["Surge", "Sway", "Heave", "Roll", "Pitch", "Yaw"]
dof = 1:2:5
position_units = [u"m", u"m", u"m", rad, rad, rad]
mass = (body_mass[dof,dof] .+ addedMassInf[dof, dof])
inv_mass_units = 1 ./ Unitful.unit.(mass')
inv_mass = inv(Unitful.ustrip.(mass)) .* inv_mass_units

# Hydrostatics calculations
g = zeros(6) .* position_units / u"s^2"
g[3] = -hydro.g[1] * u"m/s^2"
force_gravity = diag(body_mass) .* g
rho = hydro.rho[1] * u"kg/m^3"
volume = hydro.volume * u"m^3"
force_buoyancy = - rho * g * volume
CGCB = hydro.cb*u"m" - cg
force_buoyancy[4:6] = cross(CGCB, force_buoyancy[1:3])
force_hydrostatic = force_gravity + force_buoyancy

# Initial conditions
x₀ = zeros(size(dof)) .* position_units[dof]
dx₀ = zeros(size(dof)) .* position_units[dof] .* u"1/s"
tspan = [t0, tf]

# PTO damping
pto_damping = diagm(1.0e5 .* ones(size(dof))) .* radDampingUnits[dof,dof]
pto_damping_ul = Unitful.ustrip.(pto_damping)

3×3 Matrix{Float64}:
 100000.0       0.0       0.0
      0.0  100000.0       0.0
      0.0       0.0  100000.0

### System parameter formatting and energy functions 
All parameters for the system are stored in the tuple `p` to be input to the relevation hydrodynamic and power calculation functions.

In [6]:
# Unitful and unitless parameter groups
p_unitful = (
    (khsCoeff[dof, dof], 
        radDampingCoeff[dof, dof, peak_omega_index] .* 0,
        inv_mass,
        exCoeff[dof,:,:,:],
        force_hydrostatic[dof],
        (omega, phase, spectrum, dFrequency, t0, ramp_time),
        (Kᵣ[dof, dof, :], tᵣ),
    ), dx₀, x₀, tspan, ts, dt, i_ramp)

p_unitless = (
    (Unitful.ustrip.(khsCoeff[dof, dof]),
        Unitful.ustrip.(radDampingCoeff[dof, dof, peak_omega_index].*0),
        Unitful.ustrip.(inv_mass),
        Unitful.ustrip.(exCoeff[dof,:,:,:]),
        Unitful.ustrip.(force_hydrostatic[dof]),
        Unitful.ustrip.((omega, phase, spectrum, dFrequency, t0, ramp_time)),
        (Unitful.ustrip.(Kᵣ[dof, dof, :]), Unitful.ustrip.(tᵣ)),
    ),
    Unitful.ustrip.(dx₀),
    Unitful.ustrip.(x₀),
    Unitful.ustrip.(tspan),
    Unitful.ustrip.(ts),
    Unitful.ustrip.(dt),
    Unitful.ustrip.(i_ramp)
    )

(([0.0 0.0 0.0; 0.0 2.8009728217349667e6 1.1152678780490533e-9; 0.0 1.1152678780490533e-9 7.206113481356028e7], [0.0 -0.0 0.0; -0.0 0.0 -0.0; 0.0 -0.0 0.0], [1.2683677774844085e-6 1.0783267921268098e-21 -0.0; -5.655317474796333e-23 5.082890062841699e-7 -0.0; 0.0 0.0 4.693493592784703e-8], [2.0311041737386404e-8; 2.8001482923547793e6; 2.1285016593708406e-7;;; 5.179072573469057e-6; 2.797665062134365e6; 5.425501727529536e-5;;; 0.00013186801719200503; 2.793499390076672e6; 0.0013805005377349744;;; … ;;; 2961.423416463907; 609.279939604098; -116.27806247850049;;; -2730.4245875278366; 568.9036671246315; -1950.879191356708;;; -8358.99855510351; 555.4191684153839; -3663.187884503496;;;; -375.2554181510859; -0.3394843812692425; -3932.7180457749105;;; -1500.7369278289316; -5.411167118690884; -15721.822910428762;;; -3375.588449860431; -27.22483262514926; -35340.19594658676;;; … ;;; -10259.617906112284; -407.8930066804164; -7589.046475713282;;; -11101.64379574027; -304.62587088642647; -7150.0502458

## Solve and visualize the system response
Solve the system for the given inputs and visualize its response.

In [7]:
# Solve the hydrodynamic system
p = p_unitless
diff_eq_solution = Hydrodynamics.hydrodynamic_solver(p[2], p[3], p[5], 
    (p[1][1], 
        p[1][2] .+ pto_damping_ul, 
        p[1][3], 
        p[1][4], 
        p[1][5],
        p[1][6])
)

MethodError: MethodError: no method matching hydrodynamic_solver(::Vector{Float64}, ::Vector{Float64}, ::Vector{Float64}, ::Tuple{Matrix{Float64}, Matrix{Float64}, Matrix{Float64}, Array{Float64, 4}, Vector{Float64}, Tuple{Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}, Vector{Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}}, false}, Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}, Vector{Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}}, false}, WaveSpectra.OmnidirectionalSpectrum{Float64, Quantity{Float64, 𝐓^-1, Unitful.FreeUnits{(s^-1,), 𝐓^-1, nothing}}}, Float64, Float64, Float64}})
The function `hydrodynamic_solver` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  hydrodynamic_solver(::Any, ::Any, ::Any, !Matched::Symbol)
   @ Main.Hydrodynamics c:\Users\akeeste\Software\GitHub\julia\Jul

In [8]:
ci = Hydrodynamics.DSP.conv(Unitful.ustrip.(Kᵣ[dof,dof,:]), ones(1, length(dof), size(Kᵣ, 3)))

UndefVarError: UndefVarError: `DSP` not defined in `Main.Hydrodynamics`
Suggestion: check for spelling errors or missing imports.
Hint: DSP is loaded but not imported in the active module Main.

In [9]:
# Test the hydrodynamic oscillator with CIC function
irf_K = Kᵣ[dof, dof, :]
global Hydrodynamics.velocity_history = zeros(1, size(irf_K, 2), size(irf_K, 3))
Hydrodynamics.hydrodynamic_oscillator_cic(p[2], p[3], p[1], p[5][1])

MethodError: MethodError: no method matching hydrodynamic_oscillator_cic(::Vector{Float64}, ::Vector{Float64}, ::Tuple{Matrix{Float64}, Matrix{Float64}, Matrix{Float64}, Array{Float64, 4}, Vector{Float64}, Tuple{Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}, Vector{Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}}, false}, Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}, Vector{Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}}, false}, WaveSpectra.OmnidirectionalSpectrum{Float64, Quantity{Float64, 𝐓^-1, Unitful.FreeUnits{(s^-1,), 𝐓^-1, nothing}}}, Float64, Float64, Float64}, Tuple{Array{Float64, 3}, Array{Float64, 3}}}, ::Float64)
The function `hydrodynamic_oscillator_cic` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  hydrodynamic_oscillator_cic(::Any, ::Any, ::Any)
   @ Main.Hydrodynamics

In [10]:
# Solve the hydrodynamic system using the convolution integral
p = p_unitless
diff_eq_solution_cic = Hydrodynamics.hydrodynamic_solver_cic(p[2], p[3], p[5], 
    (p[1][1], 
        p[1][2].*0.0 .+ pto_damping_ul, 
        p[1][3], 
        p[1][4], 
        p[1][5],
        p[1][6],
        p[1][7],
        )
)

MethodError: MethodError: no method matching hydrodynamic_solver_cic(::Vector{Float64}, ::Vector{Float64}, ::Vector{Float64}, ::Tuple{Matrix{Float64}, Matrix{Float64}, Matrix{Float64}, Array{Float64, 4}, Vector{Float64}, Tuple{Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}, Vector{Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}}, false}, Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}, Vector{Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}}, false}, WaveSpectra.OmnidirectionalSpectrum{Float64, Quantity{Float64, 𝐓^-1, Unitful.FreeUnits{(s^-1,), 𝐓^-1, nothing}}}, Float64, Float64, Float64}, Tuple{Array{Float64, 3}, Array{Float64, 3}}})
The function `hydrodynamic_solver_cic` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  hydrodynamic_solver_cic(::Any, ::Any, ::Any)
   @ Main.Hydrodynamics c:\

In [11]:
# Visualize response
p1 = Plots.plot(diff_eq_solution, idxs = [4], label = ["Surge"], title = "Excited hydrodynamic oscillator", xaxis="Time", yaxis = "Position", lw = 2)
p1 = Plots.plot!(diff_eq_solution_cic, idxs = [4], label = ["Surge - CIC"], title = "Excited hydrodynamic oscillator", xaxis="Time", yaxis = "Position", lw = 2, ls=:dash)
p2 = Plots.plot(diff_eq_solution, idxs = [5], label = ["Heave"], title = "Excited hydrodynamic oscillator", xaxis="Time", yaxis = "Position", lw = 2)
p2 = Plots.plot!(diff_eq_solution_cic, idxs = [5], label = ["Heave - CIC"], title = "Excited hydrodynamic oscillator", xaxis="Time", yaxis = "Position", lw = 2, ls=:dash)
p3 = Plots.plot(diff_eq_solution, idxs = [6], label = ["Pitch"], title = "Excited hydrodynamic oscillator", xaxis="Time", yaxis = "Position", lw = 2)
p3 = Plots.plot!(diff_eq_solution_cic, idxs = [6], label = ["Pitch - CIC"], title = "Excited hydrodynamic oscillator", xaxis="Time", yaxis = "Position", lw = 2, ls=:dash)

Plots.plot(p1, p2, p3, layout=(3,1))

UndefVarError: UndefVarError: `diff_eq_solution` not defined in `Main`
Suggestion: add an appropriate import or assignment. This global was declared but not assigned.

In [12]:
# Calculation of power and loss
function power_performance(pto_damping, p)
    (p2, dx₀, x₀, tspan, ts, dt, i_ramp) = p

    # Unwrap, edit and rewrap `p` to combine pto and radiation damping. 
    # p2[2] .= p2[2] .+ pto_damping # Don't do this - it overwrites the base p value, skewing iterative calls to power_performance
    (k, c, inv_mass, exCoeff, constant_forces, wave, cic) = p2
    c = c .+ pto_damping
    p2 = (k, c, inv_mass, exCoeff, constant_forces, wave, cic)
    
    diff_eq_solution = Hydrodynamics.hydrodynamic_solver_cic(dx₀, x₀, ts, p2)

    # only absorb power in heave
    heave_ind = findall(x->x==3, Vector(dof))[1]
    heave_damping = pto_damping[heave_ind, heave_ind]
    heave_vel = diff_eq_solution[heave_ind,:]
    power = heave_vel .^ 2 .* heave_damping
    energy = sum(power[i_ramp:end]) * dt
    return energy
end

function power_performance(pto_damping)
    power_performance(pto_damping, p)
end

function power_loss(pto_damping, p)
    # Note p /must/ be passed into power_performance. 
    # If not passed, power_performance uses the global 'p' instead of the 'p' redefined within this function call
    - power_performance(pto_damping, p)
end

function power_loss(pto_damping)
    - power_performance(pto_damping)
end

power_loss (generic function with 2 methods)

In [13]:
### Calculate energy
p = p_unitless
energy_ul = power_performance(pto_damping_ul)
loss_ul = power_loss(pto_damping_ul, p_unitless)
[energy_ul loss_ul]
# [energy loss; energy_ul loss_ul]

MethodError: MethodError: no method matching hydrodynamic_solver_cic(::Vector{Float64}, ::Vector{Float64}, ::Vector{Float64}, ::Tuple{Matrix{Float64}, Matrix{Float64}, Matrix{Float64}, Array{Float64, 4}, Vector{Float64}, Tuple{Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}, Vector{Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}}, false}, Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}, Vector{Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}}, false}, WaveSpectra.OmnidirectionalSpectrum{Float64, Quantity{Float64, 𝐓^-1, Unitful.FreeUnits{(s^-1,), 𝐓^-1, nothing}}}, Float64, Float64, Float64}, Tuple{Array{Float64, 3}, Array{Float64, 3}}})
The function `hydrodynamic_solver_cic` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  hydrodynamic_solver_cic(::Any, ::Any, ::Any)
   @ Main.Hydrodynamics c:\

In [14]:
### Calculate gradient of energy with respect to PTO damping
p = p_unitless

# PTO_damping is not a scalar input here so there is no derivative call.
# derivative_fd = FD.derivative(power_performance, pto_damping_ul[1])

# Use gradient for multiple dofs. Also works with a single dof if pto_damping is a 1x1 array
gradient_fd = FD.gradient(power_performance, pto_damping_ul)
gradient_loss_fd = FD.gradient(power_loss, pto_damping_ul)

MethodError: MethodError: no method matching hydrodynamic_solver_cic(::Vector{Float64}, ::Vector{Float64}, ::Vector{Float64}, ::Tuple{Matrix{Float64}, Matrix{ForwardDiff.Dual{ForwardDiff.Tag{typeof(power_performance), Float64}, Float64, 9}}, Matrix{Float64}, Array{Float64, 4}, Vector{Float64}, Tuple{Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}, Vector{Quantity{Float64, 𝐀 𝐓^-1, Unitful.FreeUnits{(rad, s^-1), 𝐀 𝐓^-1, nothing}}}, false}, Base.ReinterpretArray{Float64, 1, Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}, Vector{Quantity{Float64, 𝐀, Unitful.FreeUnits{(rad,), 𝐀, nothing}}}, false}, WaveSpectra.OmnidirectionalSpectrum{Float64, Quantity{Float64, 𝐓^-1, Unitful.FreeUnits{(s^-1,), 𝐓^-1, nothing}}}, Float64, Float64, Float64}, Tuple{Array{Float64, 3}, Array{Float64, 3}}})
The function `hydrodynamic_solver_cic` exists, but no method is defined for this combination of argument types.

Closest candidates are:
